# Run fg1 — Algorithm 1 on the FULL $\gamma$ window (FLUX.1-dev, from scratch)

**The one thing this run changes.** Every previous FLUX run cut the integration window from below at the
*data scale*,

$$\gamma_{\mathrm{lo}} = \max\left(\frac{1}{S^{2}}, \frac{1}{\sigma_{\max}^{2}}\right) \approx 0.97 \qquad (S = \operatorname{std}(x_{\mathrm{refs}}) \approx 1.015),$$

following `demo_properties.ipynb` **Conclusion 1d**: below the crossover $\gamma \sim 1/\sigma_{\mathrm{data}}^{2}$ the
channel is noise-dominated, the marginal score is linear, and the integrand degenerates to the plain squared
Euclidean distance (**Conclusion 1a**),

$$\lVert s_{1} - s_{2} \rVert^{2} = \frac{\lVert x_{1} - x_{2} \rVert^{2}}{(\gamma S^{2} + 1)^{2}} \xrightarrow[\gamma \to 0]{} \lVert x_{1} - x_{2} \rVert^{2}.$$

That cut was called out in run 2's own cell 4 as the *next* variable to vary, deliberately held fixed so
$N_{\gamma}$ moved alone. **This run varies it.** The window is now the full canonical range,

$$\gamma \in [2^{-10},\, 2^{10}] \qquad (\sigma \in [2^{-5},\, 2^{5}] = [0.031,\, 32]),$$

i.e. the `logspace(-10, 10, base=2)` grid the `GlobalIEMDistance` docstring uses as its example, and the
Euclidean floor is **deliberately readmitted** rather than cut.

**What this predicts, so the result can falsify something.** Under Conclusion 1a the readmitted band adds a
$\gamma$-independent $\lVert x - x_{r} \rVert^{2}$ term weighted by $\int_{2^{-10}}^{\sim 1} d\gamma \approx 1$.
If that term dominates, $f$ degenerates towards a normalized Euclidean push-away reward and the tilt should
move particles *off* the manifold as $\lambda$ rises — visible as decoded images degrading while $E_{q}[f]$
climbs. If instead the geometry term still dominates, the low band is a cheap constant and this run should
track run 2. Cell 5 measures the split directly, before any sweeping, so the run is interpretable even if it
is later truncated.

**Not comparable to run 2 in $\sigma$ units.** Conclusion 1b's $c_{\mathrm{window}}$ rescales the whole
distance, so $\lambda_{s} = 1/\operatorname{std}_{p}(f)$ is re-measured here from a fresh 32-draw probe and
run 2's cached $\lambda_{s} = 92.195$ is **not** reused. "$m = 3$" means 3 sigma of *this* $f$.

## Configuration

| | |
|---|---|
| sampler | **Alg 1**, `adaptive_tempering_smc` + `PCNKernel`, **from scratch** (no checkpoint resume) |
| particles | $N = 8$ |
| references | $R = 64$, `RandomRefs` seed 7, uniform weights — the **same latents** run 2 used (`refs_R64_seed7.pt`), so the window is the only reward-side change |
| $\gamma$ grid | $N_{\gamma} = 30$ over $[2^{-10}, 2^{10}]$, $N_{\epsilon} = 3$, Brownian seed 123 |
| tilt | $m = 3$, $\lambda_{\max} = 3 \lambda_{s}$ with $\lambda_{s}$ measured in-run |
| generator | FLUX.1-dev, prompt `"A dog"`, guidance 1.5, 512 px, 8 Heun steps, $\sigma \in [0.002, 80]$ |
| budget | 8 h wall (`--time=480`), 1 h reserved for decode/render |

Everything else is carried over unchanged from `flux_dev_strong_tilt_sweep_2_2.ipynb`.


In [1]:
import faulthandler
import math
import os
import sys
import time

import matplotlib.pyplot as plt
import torch

T_START = time.time()

# --- hang watchdog --------------------------------------------------------------------------------
# Job 754633 held an A6000 for 6 h and produced no evidence of where it stalled. This dumps the Python
# stack of every thread to stderr (the job's .err) without killing the process, so a hang names its own
# frame. It is what identified the 754633 stall as `pipe.to(DEVICE)` paging memory-mapped safetensors
# over NFS. 30 min rather than 10: any hang worth diagnosing outlives one interval by a wide margin,
# and a tighter cadence buries the SMC's own diagnostics under tracebacks.
faulthandler.enable()
faulthandler.dump_traceback_later(1800, repeat=True, exit=False)

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isdir(os.path.join(REPO, "creativity_measure")):
    REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO not in sys.path:
    sys.path.insert(0, REPO)
RUN_DIR = os.path.join(REPO, "notebooks", "flux_fullgamma_alg1")
os.makedirs(RUN_DIR, exist_ok=True)

# --- output tagging -------------------------------------------------------------------------------
# Output names derive from N / M_TILT / SEED, which do not identify the gamma window -- and the window
# is the entire point of this run. RUN_TAG keeps a future narrow-window rerun at the same N/m/seed from
# overwriting this one, and keeps decoded_<tag>/ from mixing two runs' PNGs (the trap that forced
# 2_1 -> 2_2 in the sweep directory).
RUN_TAG = "fg1"
DECODED_DIR = os.path.join(RUN_DIR, f"decoded_{RUN_TAG}")
GRID_PNG = os.path.join(RUN_DIR, f"fullgamma_grid_{RUN_TAG}.png")

# --- wall clock -----------------------------------------------------------------------------------
# The SMC is one long library call and cannot be interrupted from outside; the kernel checks DEADLINE
# after every sweep instead. RESERVE_H is held back for VAE decoding, rendering and writing the
# notebook. Under Slurm, SLURM_JOB_END_TIME is the job's real end, so the deadline tracks --time
# automatically and WALL_BUDGET_H below is only the standalone fallback.
#
# 8 h is the requested budget. This run starts FROM SCRATCH, so it pays every fixed cost once:
# ~25 min model load (on a warm node) + ~23 min reference score bank + ~14 min lambda_s probe ~= 1 h,
# leaving ~6 h ~= 105 sweeps at the projected 3.4 min/sweep for N=8. beta=1 is plausible but not
# promised; every sweep is checkpointed and `s` is persisted, so a further leg can resume.
WALL_BUDGET_H = 8.0            # keep in sync with --time in flux_fullgamma_alg1.slurm (480 min)
RESERVE_H = 1.0
try:
    _end = float(os.environ["SLURM_JOB_END_TIME"])
except (KeyError, ValueError):
    _end = None
if _end and _end > T_START:
    DEADLINE, WALL_BUDGET_H, _src = _end - RESERVE_H * 3600, (_end - T_START) / 3600, "SLURM"
else:
    DEADLINE, _src = T_START + (WALL_BUDGET_H - RESERVE_H) * 3600, "constant"

from creativity_measure import set_default_dtype

set_default_dtype(torch.float32)          # FLUX runs bf16; everything the SMC touches is fp32
torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16

print(f"device {DEVICE} ({DTYPE})   budget {WALL_BUDGET_H:.2f} h via {_src}")
print(f"SMC deadline {time.strftime('%H:%M:%S', time.localtime(DEADLINE))}")
print(f"run tag {RUN_TAG!r}   FROM SCRATCH (no checkpoint resume)")
print(f"run dir {RUN_DIR}")


device cuda (torch.bfloat16)   budget 7.97 h via SLURM
SMC deadline 04:56:57
run tag 'fg1'   FROM SCRATCH (no checkpoint resume)
run dir /home/dcor/arielzoizner/projects/creativity-measure/notebooks/flux_fullgamma_alg1


In [2]:
from diffusers import FluxPipeline

MODEL_ID = "black-forest-labs/FLUX.1-dev"          # canonical id, kept for the CONFIG record
PROMPT = "A dog"

GUIDANCE = 1.5

IMG = 512
C, H, W = 16, IMG // 8, IMG // 8       # VAE downsamples 8x; FLUX latents have 16 channels
LATENT_DIM = C * H * W                 # 65536  (1024^2 would be 262144, and pCN mixes worse)

# --- loading is OFFLINE, and that is a deliberate change from the 2_2 notebook ---------------------
# 2_2 called huggingface_hub.auth_check() here and in the slurm preflight. Job 776480 died on exactly
# that call with a 401 GatedRepoError -- not because the run needs the hub, but because the token in
# $HOME/.hf_token is stale (776753 only got through by inheriting a valid HF_TOKEN from the submitting
# shell, which is the ambient-credential anti-pattern failure 5 in the 2_2 header warns against).
#
# Nothing in this run actually needs the hub: the full 27-file snapshot is already in $HF_HOME/hub, the
# slurm script stages it to node-local disk, and `from_pretrained` on a directory containing
# model_index.json never resolves a repo id. So the slurm script exports HF_HUB_OFFLINE=1 and treats
# auth as advisory. The assert below is the real guard -- it fails in seconds with a clear message if
# the staging step did not run, instead of letting diffusers fall back to a gated download.
MODEL_PATH = os.environ.get("FLUX_LOCAL_PATH") or MODEL_ID
if MODEL_PATH != MODEL_ID:
    assert os.path.isfile(os.path.join(MODEL_PATH, "model_index.json")), (
        f"FLUX_LOCAL_PATH={MODEL_PATH} has no model_index.json -- the slurm staging step did not "
        f"complete. Loading would fall back to the gated hub repo and 401.")
    print(f"loading from STAGED {MODEL_PATH} (offline; hub never contacted)", flush=True)
else:
    print("WARNING: FLUX_LOCAL_PATH unset -- falling back to the HF cache over NFS. This is the path "
          "that hung job 754633 for 6 h. Expect it to be slow, and expect a 401 if the cache is "
          "incomplete and HF_HUB_OFFLINE is not set.", flush=True)

pipe = FluxPipeline.from_pretrained(MODEL_PATH, torch_dtype=DTYPE).to(DEVICE)
with torch.no_grad():
    prompt_embeds, pooled_prompt_embeds, text_ids = pipe.encode_prompt(
        prompt=PROMPT, prompt_2=None, device=DEVICE, max_sequence_length=512
    )
# Free both text encoders (~9.5 GB, T5 dominates): the embeddings above are all we need.
pipe.text_encoder = pipe.text_encoder_2 = pipe.tokenizer = pipe.tokenizer_2 = None
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
transformer, vae = pipe.transformer.eval(), pipe.vae.eval()
VAE_SF, VAE_SHIFT = vae.config.scaling_factor, vae.config.shift_factor
print(f"loaded in {(time.time() - T_START) / 60:.1f} min; guidance={GUIDANCE}  "
      f"guidance_embeds={transformer.config.guidance_embeds}")
print(f"latents ({C}, {H}, {W}) = d {LATENT_DIM}  vae scaling={VAE_SF} shift={VAE_SHIFT}")


def decode(flat: torch.Tensor, chunk: int = 2) -> torch.Tensor:
    """Flat FLUX latents (B, d) -> images (B, 3, 512, 512) in [0, 1].

    Chunked because the VAE decoder, not the transformer, is the memory spike.
    """
    outs = []
    for i in range(0, flat.shape[0], chunk):
        lat = flat[i : i + chunk].reshape(-1, C, H, W).to(DEVICE, DTYPE) / VAE_SF + VAE_SHIFT
        with torch.no_grad():
            img = vae.decode(lat).sample
        outs.append(pipe.image_processor.postprocess(img.float(), output_type="pt").cpu())
    return torch.cat(outs)


loading from STAGED /tmp/arielzoizner/flux-snap-3de623fc (offline; hub never contacted)
loaded in 10.0 min; guidance=1.5  guidance_embeds=True
latents (16, 64, 64) = d 65536  vae scaling=0.3611 shift=0.1159


In [3]:
from creativity_measure.distances.edm_adapter import chunked_denoiser, edm_score_fn
from creativity_measure.generators.base import edm_generator

SIGMA_MIN, SIGMA_MAX = 2e-3, 80.0
N_STEPS = 8               # Heun ODE steps inside G (15 denoiser evals: 2/step, minus the last corrector)

# --- denoiser batch cap ---------------------------------------------------------------------------
# MEASURED, job 695266 on this exact card/model/resolution (see ../flux_lambda_sweep_strong_2/
# probe_denoiser_batch.py):
#
#     rows      24     48     96    128    192
#     empty     ok     ok     ok     ok    OOM
#     +bank     ok     ok     ok     ok    OOM      <- bank = the 1.36 GB resident ref score bank
#     free GB 15.8    8.2    2.3    0.6      -      (with the bank held)
#     s/row  0.259  0.251  0.255  0.263      -
#
# R=64 with NUM_EPS=3 needs 192 rows for the reference-bank build (score_bank -> _scores_at_gamma
# issues ONE call of NUM_EPS*R rows per gamma interval), which is exactly the width that OOMs.
# 24 rather than 128: s/row is FLAT across the range, so a narrow cap costs nothing in throughput and
# 128 would leave 0.6 GB free for an 8 h run. Fixed for the whole run, so f stays a deterministic
# function of x (CLAUDE.md invariant 1) -- chunking is mathematically identical but not bitwise.
MAX_DENOISER_ROWS = 24

# Both must be 2-D in diffusers 0.39; _prepare_latent_image_ids takes the *packed* (half) grid.
img_ids = FluxPipeline._prepare_latent_image_ids(1, H // 2, W // 2, DEVICE, DTYPE)
txt_ids = text_ids if text_ids.ndim == 2 else text_ids[0]


def denoiser(x: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    """EDM denoiser D(x_sigma, sigma) = E[X | x_sigma] backed by the FLUX transformer.

    Flow matching interpolates, EDM adds:  x_t = (1-t)x_0 + t*eps  vs  x_sigma = x_0 + sigma*eps.
    Factoring out (1-t) makes them identical under  t = sigma/(1+sigma),  x_t = x_sigma/(1+sigma).
    FLUX predicts v = eps - x_0, so  x_t - t*v = x_0  exactly -- no approximation.
    Validated by the baseline run: its lambda_eff=0 row decoded to clean, on-prompt images.

    SIGMA RANGE THIS RUN ACTUALLY VISITS. The generator still spans sigma in [0.002, 80]. The reward's
    score calls are set by the gamma window via sigma = 1/sqrt(gamma): the widened window of cell 4
    ([2^-10, 2^10]) evaluates the score at sigma in [0.031, 32], i.e. t = sigma/(1+sigma) in
    [0.030, 0.970]. Both ends sit strictly inside the trained range -- widening the window downward
    costs validity nothing, which is why the outer bound of Conclusion 1d is not the binding constraint
    here. (The narrow window run 2 used only reached down to sigma ~ 1.0.)

    Returns fp32 (the transformer runs bf16): reward.x_refs pins dtype for the whole SMC, and bf16
    weight/ESS arithmetic would quietly corrupt the beta schedule.
    """
    b = x.shape[0]
    sig = sigma.reshape(b, 1, 1, 1).to(torch.float32)
    t = (sigma / (1.0 + sigma)).to(torch.float32)
    x_t = x.to(torch.float32) / (1.0 + sig)
    packed = FluxPipeline._pack_latents(x_t.to(DTYPE), b, C, H, W)
    with torch.no_grad():
        v = transformer(
            hidden_states=packed,
            timestep=t.to(DTYPE),                                   # forward() scales by 1000 itself
            guidance=torch.full((b,), GUIDANCE, device=x.device, dtype=torch.float32),
            pooled_projections=pooled_prompt_embeds.expand(b, -1).to(DTYPE),
            encoder_hidden_states=prompt_embeds.expand(b, -1, -1).to(DTYPE),
            txt_ids=txt_ids,
            img_ids=img_ids,
            return_dict=False,
        )[0]
    v = FluxPipeline._unpack_latents(v, IMG, IMG, 8).to(torch.float32)
    return (x_t - t.reshape(b, 1, 1, 1) * v).to(torch.float32)


# Wrap ONCE and build both consumers from the wrapper: this covers the generator path
# (heun_prob_flow -> edm_ode_step) and every score path (marginal_score -> score_fn), including the
# reference score_bank build, which is the widest call in the run.
denoiser_capped = chunked_denoiser(denoiser, MAX_DENOISER_ROWS)

G = edm_generator(denoiser_capped, img_shape=(C, H, W), sigma_min=SIGMA_MIN,
                  sigma_max=SIGMA_MAX, n_steps=N_STEPS)
score_fn = edm_score_fn(denoiser_capped, (C, H, W))
print(f"generator + score ready   sigma in [{SIGMA_MIN}, {SIGMA_MAX}]  n_steps={N_STEPS}")
print(f"denoiser batch capped at {MAX_DENOISER_ROWS} rows "
      f"(measured ceiling 128, OOM at 192; s/row flat so the cap is free)")


generator + score ready   sigma in [0.002, 80.0]  n_steps=8
denoiser batch capped at 24 rows (measured ceiling 128, OOM at 192; s/row flat so the cap is free)


In [4]:
from creativity_measure import SquaredGlobalIEMDistance
from creativity_measure.tilt import NormalizedExpectedDistanceReward

# --- knobs: run 2's, except N (16 -> 8) and the gamma window ---------------------------------------
# R / N_GAMMA / NUM_EPS / the reference seed (7) / the Brownian seed (123) / MAX_DENOISER_ROWS are all
# run 2's values, so the WINDOW is the only reward-side variable moving. MAX_SWEEPS stays at 2_2's 40.
#
# N = 8 (requested). Halving the cloud halves the sweep cost (3.4 vs 7.4 min) and buys roughly twice
# the ladder in the same 8 h, at the price of a coarser ESS: with N=8 the ess_target of 0.80 is 6.4
# particles, and `uniq/N` moves in steps of 0.125. Read the degeneracy diagnostics accordingly -- a
# single lost lineage is a 12.5% drop here, not 6.25%.
R, N_GAMMA, NUM_EPS = 64, 30, 3
N = 8
MAX_SWEEPS = 40
M_TILT = 3.0
ESS_TARGET = 0.80
SEED = 101

# lambda_s = 1/std_p(f) is estimated from PROBE_SIZE held-out draws. 694278 used 8 and the estimate did
# not replicate (probe std 0.0039 vs its own level-0 cloud's 0.0140). Normal-theory SE on a std scales
# as 1/sqrt(2(n-1)): 8 -> 27%, 32 -> 13%. 32 is run 2's value and is kept.
PROBE_SIZE = 32

# --- references: LOADED, never regenerated ---------------------------------------------------------
# These are run 2's own reference latents, copied into this directory. Loading rather than regenerating
# does two things. (1) It takes G out of the reproducibility chain -- job 697271 died because
# regenerating refs through a 12B bf16 transformer on a different card shifted f by 11% of std_p(f).
# (2) More important here: it makes the reference set BIT-IDENTICAL to run 2's, so E_p[f], lambda_s and
# every downstream number differ from run 2 for exactly one reason, the gamma window. Regenerating
# would confound the comparison with round-off.
REFS_PATH = os.path.join(RUN_DIR, f"refs_R{R}_seed7.pt")
GPU_NAME = torch.cuda.get_device_name(0)
assert os.path.exists(REFS_PATH), f"missing {REFS_PATH}; copy it from ../flux_lambda_sweep_strong_2/"
_rf = torch.load(REFS_PATH, map_location="cpu", weights_only=False)
latent_refs = _rf["x_refs"].to(DEVICE)
assert _rf["R"] == R and _rf["latent_dim"] == LATENT_DIM
# Not a resume, so this is a warning rather than an abort: the latents reload exactly on any card, and
# nothing here is stitched onto a prior run's weights. But the score bank IS rebuilt on this card, so a
# different architecture makes the numbers non-comparable to run 2 at the 11%-of-std level.
if _rf.get("gpu") != GPU_NAME:
    print(f"*** WARNING: refs were generated on {_rf.get('gpu')!r}, this job is on {GPU_NAME!r}. "
          f"The run is self-consistent, but bf16 kernels differ across architectures, so comparisons "
          f"to run 2 carry ~11% of std_p(f) of drift. Prefer --constraint=a6000.")
S = latent_refs.std().item()
print(f"references LOADED from {os.path.basename(REFS_PATH)}  {tuple(latent_refs.shape)}  "
      f"S={S:.4f}   card {GPU_NAME}")

# ================== THE CHANGE: the full gamma window ==============================================
# Run 2:   GAMMA_LO = max(1/S^2, 1/SIGMA_MAX^2) = 0.9713   -- the Conclusion-1d lower cut at the data
#          scale, which drops the noise-dominated band where the integrand degenerates to the plain
#          squared Euclidean distance (Conclusion 1a).
# Here:    GAMMA_LO = 2^-10                                -- the floor is READMITTED on purpose. This
#          is the `logspace(-10, 10, base=2)` grid GlobalIEMDistance's own docstring uses as its
#          example, and it is the window variation run 2's cell 4 named as the next thing to try.
#
# 2^-10 is a choice, not a limit: the denoiser stays valid down to gamma = 1/SIGMA_MAX^2 = 1.56e-4
# (= 2^-12.68), asserted below. 2^-10 keeps the grid symmetric about gamma = 1 and sits ~2.7 octaves
# inside the validity edge, so no score is ever evaluated at an extrapolated sigma.
#
# WHAT IT COSTS: nothing in rows -- a reward call is (N_GAMMA-1)*NUM_EPS*B regardless of where the
# nodes sit. What it costs is RESOLUTION. N_GAMMA=30 now spans 20 octaves instead of ~10, so the
# left-endpoint rule integrates at 0.69 octaves/node against run 2's 0.35, and half the grid (15 of 30
# nodes) now sits in the readmitted band. demo_properties Concl. 2 calibrated "N_gamma >= 25 suffices"
# ON THE CUT WINDOW, so that floor does NOT transfer unexamined to this one. N_GAMMA is held at 30
# anyway, deliberately: doubling it would double every reward call and confound the window with the
# grid, which is the mistake run 2 explicitly avoided in the other direction. Cell 5 measures the
# per-interval profile so the discretization error is observable rather than assumed, and a follow-up
# run at N_GAMMA=60 is the clean way to bound it.
GAMMA_LO = 2.0**-10
GAMMA_HI = min(2.0**10, 1.0 / SIGMA_MIN**2)
assert GAMMA_LO >= 1.0 / SIGMA_MAX**2, (
    f"gamma_lo={GAMMA_LO:.4g} is below the denoiser's valid floor 1/sigma_max^2="
    f"{1.0 / SIGMA_MAX**2:.4g}; the learned score is not defined there")
GAMMAS = torch.logspace(math.log2(GAMMA_LO), math.log2(GAMMA_HI), N_GAMMA, base=2).to(DEVICE)
GAMMA_LO_RUN2 = max(1.0 / S**2, 1.0 / SIGMA_MAX**2)      # 0.9713 -- what this run replaces

_oct = math.log2(GAMMA_HI / GAMMA_LO)
_oct2 = math.log2(GAMMA_HI / GAMMA_LO_RUN2)
print(f"\ngamma window [{GAMMA_LO:.4g}, {GAMMA_HI:.4g}] = [2^{math.log2(GAMMA_LO):.0f}, "
      f"2^{math.log2(GAMMA_HI):.0f}]  ->  sigma in [{GAMMA_HI**-0.5:.4g}, {GAMMA_LO**-0.5:.4g}]")
print(f"  {N_GAMMA} nodes over {_oct:.1f} octaves = {_oct / (N_GAMMA - 1):.2f} octaves/interval")
print(f"  run 2 used [{GAMMA_LO_RUN2:.4g}, {GAMMA_HI:.4g}] = {_oct2:.1f} octaves at "
      f"{_oct2 / (N_GAMMA - 1):.2f} octaves/interval")
print(f"  READMITTED band: gamma < {GAMMA_LO_RUN2:.4g}, i.e. "
      f"{sum(1 for g in GAMMAS.tolist() if g < GAMMA_LO_RUN2)} of {N_GAMMA} nodes; it holds "
      f"{100 * (GAMMA_LO_RUN2 - GAMMA_LO) / (GAMMA_HI - GAMMA_LO):.3f}% of the raw d-gamma measure")
print(f"  (raw d-gamma share is NOT the contribution share -- the integrand is far from flat. Cell 5.)")

D2 = SquaredGlobalIEMDistance(None, GAMMAS, num_eps=NUM_EPS, seed=123, score_fn=score_fn)

t0 = time.time()
reward = NormalizedExpectedDistanceReward(distance=D2, x_refs=latent_refs)   # builds the ref bank
t_norm = time.time() - t0
_bank_rows = (N_GAMMA - 1) * NUM_EPS * R
print(f"\nreference score bank built in {t_norm:.0f}s from {_bank_rows} rows "
      f"({t_norm / _bank_rows:.3f} s/row; A6000 measured 0.251)")
print(f"  bank VRAM held for the whole run: (N_GAMMA-1)*NUM_EPS*R*d*4 = "
      f"{(N_GAMMA - 1) * NUM_EPS * R * LATENT_DIM * 4 / 2**30:.2f} GB")

# --- free correctness check: mean_r f(x_r) == (R-1)/R exactly --------------------------------------
# The numerator averages R terms INCLUDING the zero self-pair; reference_pair_mean divides by R(R-1).
# With uniform weights the identity is exact, and it costs ZERO score rows -- pairwise() aliases
# batch_scores to the cached ref bank when X is x_refs (address identity), so the zero diagonal is
# bitwise-free.
#
# IT IS AN IDENTITY ON THE MEAN, NOT PER ELEMENT, and asserting per element would be wrong:
#     mean_r f(x_r) = [sum_{i,j} D2 / R^2] * R(R-1) / sum_{i!=j} D2 = (R-1)/R,
# while an individual f(x_r) equals (R-1)/R only if every row sum of D2 is equal. It is not -- the
# spread of those row sums IS the spread of f under p, i.e. the quantity lambda_s measures below, and
# it is emphatically nonzero. So the tight assert goes on the mean (1e-6: this is arithmetic, not
# estimation) and the per-reference spread is REPORTED as a first look at std_p(f).
# The check tests normalization wiring only -- it says nothing about score-field determinism -- but it
# is the cheapest way to catch a mis-specified window breaking the numerator/denominator pairing, and
# a mean landing on exactly 1.0 would mean the self-pair was dropped from the numerator.
f_refs = reward(latent_refs)
_expect = (R - 1) / R
print(f"\nf(refs): mean {float(f_refs.mean()):.8f}   expected (R-1)/R = {_expect:.8f}   "
      f"|dev| {abs(float(f_refs.mean()) - _expect):.2e}")
print(f"  per-reference spread: std {float(f_refs.std()):.4f}, "
      f"range [{float(f_refs.min()):.4f}, {float(f_refs.max()):.4f}]  (nonzero by construction)")
assert abs(float(f_refs.mean()) - _expect) < 1e-6, (
    f"mean f(refs) = {float(f_refs.mean()):.8f} != (R-1)/R = {_expect:.8f}; the normalizer and the "
    f"numerator disagree. Landing on exactly 1.0 means the self-pair was not included in the mean.")

# --- lambda scale, measured on the SAME 32 held-out draws run 2 used -------------------------------
# The generator is re-seeded to 7 and the first R rows are drawn and DISCARDED, which advances the
# stream to exactly where run 2 was after it generated its references. The next PROBE_SIZE rows are
# therefore run 2's probe latents, bit-for-bit. Without the discard the probe would be the first 32 z's
# of the REFERENCE draw -- i.e. f evaluated on the references themselves, pinned so that its mean is
# (R-1)/R by the identity above, and lambda_s would describe the wrong distribution entirely.
#
# NOT COMPARABLE TO RUN 2 IN SIGMA UNITS. Concl. 1b: c_window = sqrt(int dgamma/g^2) rescales the whole
# distance, so "m = 3" is 3 sigma of THIS f. Run 2 measured lambda_s = 92.195 (std_p(f) = 0.010846) on
# these same 32 draws with the cut window. Expect a different value; that is the measurement, not a bug.
LAM_S_RUN2 = 92.195
gen = torch.Generator(device="cpu").manual_seed(7)
_discard = torch.randn(R, LATENT_DIM, generator=gen)          # advance past the reference draw
z_probe = torch.randn(PROBE_SIZE, LATENT_DIM, generator=gen)
del _discard
t0 = time.time()
with torch.no_grad():
    x_probe = G(z_probe.to(DEVICE))
    f_probe = reward(x_probe)
t_probe = time.time() - t0
# The probe must be held out. If the discard above were dropped these would coincide with the refs and
# the minimum distance would be exactly 0.
_dmin = float(torch.cdist(x_probe[:4], latent_refs).min())
assert _dmin > 0, "probe latents coincide with references -- the generator stream was not advanced"

f_std = float(f_probe.std())
lam_s = (1.0 / f_std) if f_std > 0 else float("inf")
LAM_MAX = M_TILT * lam_s
print(f"\nf ~ p on {PROBE_SIZE} held-out draws ({t_probe / 60:.1f} min):")
print(f"  mean={float(f_probe.mean()):.4f}  std={f_std:.4f}  CV={f_std / float(f_probe.mean()):.2%}  "
      f"range=[{float(f_probe.min()):.4f}, {float(f_probe.max()):.4f}]")
print(f"  min dist(probe, refs) = {_dmin:.4g} > 0, so the probe is held out")
print(f"lambda_s = 1/std_p(f) = {lam_s:.2f}   (run 2, cut window, same 32 draws: {LAM_S_RUN2:.2f}; "
      f"ratio {lam_s / LAM_S_RUN2:.2f}x)")
print(f"LAM_MAX = {M_TILT:.0f} * lambda_s = {LAM_MAX:.1f}")

print(f"\n{'m':>5} {'lambda':>10} {'ESS/N':>7}   (one-shot IS from p to q_lambda on the probe)")
for m in sorted({1.0, 2.0, M_TILT, 5.0, 10.0}):
    w = torch.softmax(m * lam_s * f_probe, dim=0)
    print(f"{m:5.1f} {m * lam_s:10.1f} {float(1.0 / (w.pow(2).sum() * PROBE_SIZE)):7.2f}"
          + ("   <- this run" if m == M_TILT else ""))

# --- cost projection, from the per-row cost measured seconds ago -----------------------------------
# Rows are denoiser forward passes. Per particle: G(z) costs 2*N_STEPS-1 (Heun), the reward costs
# (N_GAMMA-1)*NUM_EPS with the refs cached -- 15 vs 87, so the reward is ~85% of a sweep and trimming
# N_STEPS is not worth it. Both previous overruns came from projecting on a constant carried over from
# an older job, so this uses t_probe, measured on this card at this configuration.
GEN_ROWS = 2 * N_STEPS - 1
REW_ROWS = (N_GAMMA - 1) * NUM_EPS
s_row = t_probe / (PROBE_SIZE * (GEN_ROWS + REW_ROWS))
t_sweep = N * (GEN_ROWS + REW_ROWS) * s_row
budget_s = (WALL_BUDGET_H - RESERVE_H) * 3600 - (time.time() - T_START)
sweeps_afford = budget_s / t_sweep
print(f"\nmeasured {s_row:.3f} s/row  ({GEN_ROWS} G + {REW_ROWS} reward rows per particle)")
print(f"  sweep of N={N}: {t_sweep / 60:.1f} min   budget left {budget_s / 3600:.2f} h "
      f"-> ~{sweeps_afford:.0f} sweeps")
for lbl, swl in (("optimistic (695271's 13.5 sweeps/level)", 13.5),
                 ("likely (20 sweeps/level as acceptance falls)", 20.0),
                 ("pessimistic (every level to the cap)", float(MAX_SWEEPS))):
    print(f"    {lbl:46s} -> {sweeps_afford / swl:4.1f} levels")
print("  Truncation is accepted: every sweep is checkpointed and `s` is persisted, so a further leg "
      "can resume from partial_*.pt.")


references LOADED from refs_R64_seed7.pt  (64, 65536)  S=1.0147   card NVIDIA RTX A6000

gamma window [0.0009766, 1024] = [2^-10, 2^10]  ->  sigma in [0.03125, 32]
  30 nodes over 20.0 octaves = 0.69 octaves/interval
  run 2 used [0.9713, 1024] = 10.0 octaves at 0.35 octaves/interval
  READMITTED band: gamma < 0.9713, i.e. 15 of 30 nodes; it holds 0.095% of the raw d-gamma measure
  (raw d-gamma share is NOT the contribution share -- the integrand is far from flat. Cell 5.)

reference score bank built in 1429s from 5568 rows (0.257 s/row; A6000 measured 0.251)
  bank VRAM held for the whole run: (N_GAMMA-1)*NUM_EPS*R*d*4 = 1.36 GB

f(refs): mean 0.98437500   expected (R-1)/R = 0.98437500   |dev| 0.00e+00
  per-reference spread: std 0.0151, range [0.9620, 1.0452]  (nonzero by construction)

f ~ p on 32 held-out draws (13.7 min):
  mean=0.9985  std=0.0127  CV=1.27%  range=[0.9794, 1.0386]
  min dist(probe, refs) = 258.8 > 0, so the probe is held out
lambda_s = 1/std_p(f) = 78.68   (run 2

In [5]:
# =================================================================================================
# THE DIAGNOSTIC THIS RUN EXISTS FOR: where in gamma does D^2_IEM actually come from?
# =================================================================================================
# Everything above only widened the window. This measures what widening it BOUGHT, before a single
# sweep is spent, so the run answers its question even if the SMC is later truncated by the deadline.
#
# `iem_sq_increments_one_to_many` returns the per-interval contributions (already multiplied by
# dgamma) for one reference against a batch, so summing over axis 0 reproduces D^2_IEM exactly. It is
# the "NOT used in practice" path -- it re-scores the batch instead of reusing the cached bank -- which
# is fine at this size: (N_GAMMA-1)*NUM_EPS*(1+G_DIAG) = 783 rows, ~3 min, paid once.
#
# THREE THINGS IT DECIDES:
#   1. What fraction of the integral the READMITTED band (gamma < 0.97) carries. If it is negligible,
#      this run is run 2 with extra steps and the negative result is cheap. If it dominates, f has been
#      converted into a Euclidean push-away reward and the decoded images at high lambda are the test.
#   2. Whether the low band behaves as Conclusion 1a predicts. 1a is a claim with a closed form:
#      ||s1 - s2||^2 = ||x1 - x2||^2 / (gamma S^2 + 1)^2. The predicted column below is that formula on
#      the measured ||x1 - x2||^2, with NO fitted constant. Agreement confirms the mechanism; a large
#      gap means the FLUX score does something else in the noise-dominated regime and Concl. 1a (derived
#      on a GMM) does not transfer to this p.
#   3. Whether N_GAMMA=30 still resolves the integrand over 20 octaves. A profile whose peak is spanned
#      by one or two intervals is a discretization warning, and the follow-up is N_GAMMA=60.
from creativity_measure.distances.global_iem import iem_sq_increments_one_to_many
from creativity_measure.distances.utils import simulate_brownian

G_DIAG = 8                                   # reference pairs (ref 0 vs refs 1..8); 783 rows total
t0 = time.time()
_gam = GAMMAS.to(device=DEVICE, dtype=latent_refs.dtype)
_W = simulate_brownian(_gam, NUM_EPS, LATENT_DIM, 123, DEVICE, latent_refs.dtype)
with torch.no_grad():
    inc = iem_sq_increments_one_to_many(
        latent_refs[0:1], latent_refs[1:1 + G_DIAG], _W, _gam, None, score_fn
    )                                        # (N_GAMMA-1, NUM_EPS, G_DIAG), dgamma already folded in
t_diag = time.time() - t0

prof = inc.mean(dim=(1, 2)).double().cpu()               # (N_GAMMA-1,) mean contribution per interval
total = float(prof.sum())
cum = torch.cumsum(prof, 0) / total
g_left = GAMMAS[:-1].double().cpu()
dg = (GAMMAS[1:] - GAMMAS[:-1]).double().cpu()

# Conclusion 1a's closed form, with no fitted constant: ||x1-x2||^2 * dgamma / (gamma S^2 + 1)^2.
dx2 = (latent_refs[0:1] - latent_refs[1:1 + G_DIAG]).pow(2).sum(-1).mean().double().cpu()
pred_1a = dx2 * dg / (g_left * S**2 + 1.0).pow(2)

readmit = g_left < GAMMA_LO_RUN2
frac_readmit = float(prof[readmit].sum()) / total
n_rows_diag = (N_GAMMA - 1) * NUM_EPS * (1 + G_DIAG)
print(f"per-interval IEM^2 profile, ref 0 vs refs 1..{G_DIAG}  "
      f"({t_diag / 60:.1f} min, {n_rows_diag} rows)")
print(f"  E[||x_i - x_j||^2] over those pairs = {float(dx2):.4g}  (d = {LATENT_DIM}, so ~2d = "
      f"{2 * LATENT_DIM} for independent unit-scale latents)")
print(f"\n{'gamma':>10} {'sigma':>8} {'d-gamma':>10} {'contrib':>12} {'share':>7} {'cum':>7} "
      f"{'Concl.1a':>12} {'ratio':>7}")
for i in range(N_GAMMA - 1):
    if bool(readmit[i]):
        mark = "  <- readmitted"
    elif i > 0 and bool(readmit[i - 1]):
        mark = "  <- run 2's window starts here"
    else:
        mark = ""
    print(f"{float(g_left[i]):10.4g} {float(g_left[i]) ** -0.5:8.3g} {float(dg[i]):10.4g} "
          f"{float(prof[i]):12.5g} {float(prof[i]) / total:7.2%} {float(cum[i]):7.2%} "
          f"{float(pred_1a[i]):12.5g} {float(prof[i] / pred_1a[i]):7.3f}{mark}")

print(f"\nREADMITTED BAND (gamma < {GAMMA_LO_RUN2:.4g}, the region run 2 cut):")
print(f"  carries {frac_readmit:.2%} of D^2_IEM, over {int(readmit.sum())} of {N_GAMMA - 1} intervals")

# Is the readmitted band the Euclidean floor Conclusion 1a predicts? No fitted constant, so a ratio
# near 1 is a real confirmation and a ratio far from 1 falsifies the transfer of 1a to this p.
lo_ratio = float((prof[readmit] / pred_1a[readmit]).median())
if 0.5 < lo_ratio < 2.0:
    verdict_1a = "CONFIRMS the Euclidean floor -- f now carries a ||x - x_r||^2 term"
else:
    verdict_1a = ("DOES NOT match 1a. The FLUX score is not linear in the noise-dominated band, so "
                  "1a (derived on a GMM) does not transfer to this p. Read the profile before "
                  "trusting any conclusion about what the tilt rewards.")
print(f"  median measured/predicted ratio there: {lo_ratio:.3f} -> {verdict_1a}")

peak = int(prof.argmax())
peak_share = float(prof[peak]) / total
if peak_share < 0.5:
    verdict_res = "RESOLVED"
else:
    verdict_res = ("UNDER-RESOLVED -- a single interval holds over half the integral; the follow-up "
                   "is N_GAMMA=60 on this same window")
print(f"  peak interval: gamma={float(g_left[peak]):.4g} "
      f"(sigma={float(g_left[peak]) ** -0.5:.3g}) carrying {peak_share:.2%} -> {verdict_res}")

torch.save({"gammas": GAMMAS.cpu(), "profile": prof, "pred_1a": pred_1a, "dx2": dx2,
            "frac_readmitted": frac_readmit, "gamma_lo_run2": GAMMA_LO_RUN2, "G_diag": G_DIAG,
            "num_eps": NUM_EPS, "S": S, "lo_ratio": lo_ratio, "peak_share": peak_share},
           os.path.join(RUN_DIR, f"gamma_profile_{RUN_TAG}.pt"))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].loglog(g_left, prof, "o-", label="measured")
ax[0].loglog(g_left, pred_1a, "s--", alpha=0.7,
             label=r"Concl. 1a: $\|\Delta x\|^2 d\gamma/(\gamma S^2+1)^2$")
ax[0].axvline(GAMMA_LO_RUN2, color="crimson", ls=":", label=r"run 2's $\gamma_{lo}$")
ax[0].set_xlabel(r"$\gamma$"); ax[0].set_ylabel(r"contribution to $D^2_{IEM}$")
ax[0].set_title("per-interval contribution"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].semilogx(g_left, cum, "o-")
ax[1].axvline(GAMMA_LO_RUN2, color="crimson", ls=":", label=r"run 2's $\gamma_{lo}$")
ax[1].axhline(frac_readmit, color="grey", ls="--", label=f"readmitted = {frac_readmit:.1%}")
ax[1].set_xlabel(r"$\gamma$"); ax[1].set_ylabel("cumulative fraction"); ax[1].set_ylim(0, 1.02)
ax[1].set_title("where the integral accumulates"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RUN_DIR, f"gamma_profile_{RUN_TAG}.png"), dpi=110, bbox_inches="tight")
plt.show()

del inc, _W
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


per-interval IEM^2 profile, ref 0 vs refs 1..8  (3.3 min, 783 rows)
  E[||x_i - x_j||^2] over those pairs = 1.126e+05  (d = 65536, so ~2d = 131072 for independent unit-scale latents)

     gamma    sigma    d-gamma      contrib   share     cum     Concl.1a   ratio
 0.0009766       32  0.0005985       43.946   0.01%   0.01%       67.257   0.653  <- readmitted
  0.001575     25.2  0.0009654       67.032   0.02%   0.03%       108.34   0.619  <- readmitted
   0.00254     19.8   0.001557       97.901   0.03%   0.06%        174.4   0.561  <- readmitted
  0.004098     15.6   0.002511       146.81   0.04%   0.09%        280.4   0.524  <- readmitted
  0.006609     12.3   0.004051       218.13   0.06%   0.15%       449.93   0.485  <- readmitted
   0.01066     9.69   0.006533       333.65   0.09%   0.24%       719.72   0.464  <- readmitted
   0.01719     7.63    0.01054       501.63   0.13%   0.37%       1145.5   0.438  <- readmitted
   0.02773     6.01      0.017        763.7   0.20%   0.58%    

In [6]:
from creativity_measure import LevelSnapshot, PCNKernel, adaptive_tempering_smc_sample

CONFIG = {
    "model_id": MODEL_ID, "img": IMG, "C": C, "H": H, "W": W, "latent_dim": LATENT_DIM,
    "vae_scaling_factor": VAE_SF, "vae_shift_factor": VAE_SHIFT,
    "N": N, "m_tilt": M_TILT, "lam_max": LAM_MAX, "lam_s": lam_s, "ess_target": ESS_TARGET,
    "n_mcmc": None, "max_n_mcmc": MAX_SWEEPS, "seed": SEED, "prompt": PROMPT,
    "guidance": GUIDANCE, "n_steps": N_STEPS, "sigma_min": SIGMA_MIN, "sigma_max": SIGMA_MAX,
    "R": R, "n_gamma": N_GAMMA, "num_eps": NUM_EPS,
    # The window, spelled out. `gamma_lo_run2` is recorded next to it so a later reader can see at a
    # glance which window a checkpoint belongs to without re-deriving it from S.
    "gamma_lo": GAMMA_LO, "gamma_hi": GAMMA_HI, "gamma_lo_run2": GAMMA_LO_RUN2,
    "full_gamma_window": True, "frac_readmitted": frac_readmit, "concl_1a_ratio": lo_ratio,
    "probe_size": PROBE_SIZE, "f_std_p": f_std, "f_mean_p": float(f_probe.mean()),
    "data_scale_S": S, "max_denoiser_rows": MAX_DENOISER_ROWS,
    "t_norm_s": t_norm, "t_probe_s": t_probe, "t_diag_s": t_diag, "s_row": s_row,
    "t_sweep_s": t_sweep, "run_tag": RUN_TAG, "resumed_from": None, "gpu": GPU_NAME,
}
PARTIAL_CKPT = os.path.join(RUN_DIR, f"partial_N{N}_m{M_TILT:.0f}_seed{SEED}_{RUN_TAG}.pt")


class DeadlineReached(RuntimeError):
    """Raised by the kernel when the wall-clock deadline passes, to stop the SMC cleanly."""


class CheckpointingPCNKernel(PCNKernel):
    """Persists the cloud every sweep and stops the run at a wall-clock deadline.

    Only *reads* state after `super().step()` has run, so no randomness is consumed and no value
    changes: the run is bit-identical to one using a plain PCNKernel (verified on the 2D toy against
    X, logw, betas, ess_history, n_mcmc_history and stop_reason_history).

    Keying by `beta` is the trick -- beta changes only between levels, so each entry is overwritten
    within a level and ends up holding that level's LAST sweep, i.e. exactly the post-rejuvenation
    cloud the final snapshot records. `init` is overridden too, so the beta=0 anchor is captured.
    Writes are temp-file + os.replace, so a kill mid-write cannot corrupt the last good file.

    `ancestors` is arange here (resampling happens outside the kernel); the final save has the real ones.
    `state.ess` is the level's post-reweight ESS, set by the SMC loop before resampling -- reading it
    here is what lets a deadline-truncated run report ESS/N instead of the NaN column the run-2
    notebook was stuck with.
    `self.s` is the adapted pCN step size. It is persisted because the source-run defect that forced
    2_2 to raise `adapt_rate` was precisely a checkpoint without it: a resume then restarts from s0 and
    re-learns the local geometry from nothing. `adapt_rate` is left at the library default of 0.1 here
    -- 2_2's 0.25 was a workaround for that missing state, not a better setting.
    """

    def __init__(self, *a, path, config, deadline, save_z=True, **kw):
        super().__init__(*a, **kw)
        self._path, self._config, self._deadline, self._save_z = path, config, deadline, save_z
        self._latest, self._sweeps_at, self._sweeps = {}, {}, 0

    def _record(self, state, *, beta, lam, acc):
        e = {"beta": beta, "lam_eff": beta * lam,
             "X": state.X.detach().to("cpu").clone(), "fX": state.fX.detach().to("cpu").clone(),
             "logw": state.logw.detach().to("cpu").clone(),
             "ess": float(getattr(state, "ess", float("nan"))),
             "s": float(self.s),
             "ancestors": torch.arange(state.X.shape[0]),
             "sweeps_at_beta": self._sweeps_at.get(beta, 0), "acc": acc}
        if self._save_z:
            e["aux"] = {k: v.detach().to("cpu").clone() for k, v in state.aux.items()}
        self._latest[beta] = e
        tmp = f"{self._path}.tmp"
        torch.save({"levels": list(self._latest.values()), "betas": list(self._latest.keys()),
                    "config": self._config, "partial": True, "sweeps_done": self._sweeps}, tmp)
        os.replace(tmp, self._path)

    def init(self, n_particles, f, *, generator, device, dtype):
        st = super().init(n_particles, f, generator=generator, device=device, dtype=dtype)
        self._record(st, beta=0.0, lam=0.0, acc=float("nan"))
        return st

    def step(self, state, *, beta, lam, f, generator):
        acc = super().step(state, beta=beta, lam=lam, f=f, generator=generator)
        b = float(beta)
        self._sweeps += 1
        self._sweeps_at[b] = self._sweeps_at.get(b, 0) + 1
        self._record(state, beta=b, lam=float(lam), acc=float(acc))
        if time.time() > self._deadline:
            raise DeadlineReached(f"deadline after {self._sweeps} sweeps at beta={b:.4f}; "
                                  f"{len(self._latest)} levels on disk")
        return acc


kernel = CheckpointingPCNKernel(G, LATENT_DIM, path=PARTIAL_CKPT, config=CONFIG, deadline=DEADLINE)
print(f"SMC (from scratch): N={N} R={R} N_gamma={N_GAMMA} gamma=[{GAMMA_LO:.4g}, {GAMMA_HI:.4g}] "
      f"m={M_TILT:.0f} lam={LAM_MAX:.1f} ess_target={ESS_TARGET} "
      f"n_mcmc=None (adaptive, cap {MAX_SWEEPS})")
print(f"pCN: s0={kernel.s:.3f} target_acc={kernel.target_acc} adapt_rate={kernel.adapt_rate}")
print(f"projected {t_sweep / 60:.1f} min/sweep; deadline "
      f"{time.strftime('%H:%M:%S', time.localtime(DEADLINE))}")
print(f"checkpointing every sweep -> {os.path.basename(PARTIAL_CKPT)}", flush=True)

t0, timed_out = time.time(), False
try:
    res = adaptive_tempering_smc_sample(
        reward, lam=LAM_MAX, n_particles=N, kernel=kernel, ess_target=ESS_TARGET,
        n_mcmc=None, max_n_mcmc=MAX_SWEEPS, keep_levels=True, snapshot_device="cpu", seed=SEED,
    )
    levels = res.levels
    assert levels is not None
except DeadlineReached as exc:
    timed_out = True
    print(f"\n*** STOPPED ON DEADLINE: {exc}\n*** Rendering the levels that completed.")
    _e = list(kernel._latest.values())
    levels = [LevelSnapshot(beta=x["beta"], lam_eff=x["lam_eff"], X=x["X"], fX=x["fX"],
                            logw=x["logw"], ancestors=x["ancestors"],
                            ess=x.get("ess", float("nan"))) for x in _e]

    class _PartialResult:
        """Stands in for the result dataclass. ESS is read back from the kernel's per-level records
        (`state.ess`, set by the SMC loop before resampling). Stop reasons are INFERRED: all sweeps
        used -> 'cap', fewer -> 'decorr', and the level in progress when the deadline hit -> 'timeout'."""

        def __init__(s, entries):
            tail = entries[1:]
            s.betas = [x["beta"] for x in tail]
            s.ess_history = [x.get("ess", float("nan")) for x in tail]
            s.acc_history = [x["acc"] for x in tail]
            s.n_mcmc_history = [x["sweeps_at_beta"] for x in tail]
            s.stop_reason_history = [("cap" if x["sweeps_at_beta"] >= MAX_SWEEPS else "decorr")
                                     for x in tail]
            if s.stop_reason_history:
                s.stop_reason_history[-1] = "timeout"
            s.X, s.logw, s.levels = levels[-1].X, levels[-1].logw, levels

    res = _PartialResult(_e)

mins = (time.time() - t0) / 60
ckpt = os.path.join(RUN_DIR,
                    f"levels_N{N}_m{M_TILT:.0f}_seed{SEED}_{RUN_TAG}"
                    f"{'_TIMEOUT' if timed_out else ''}.pt")
torch.save({
    "levels": [{"beta": s.beta, "lam_eff": s.lam_eff, "X": s.X, "fX": s.fX,
                "logw": s.logw, "ancestors": s.ancestors, "ess": s.ess} for s in levels],
    "X": res.X, "logw": res.logw, "betas": res.betas,
    "ess_history": res.ess_history, "acc_history": res.acc_history,
    "n_mcmc_history": res.n_mcmc_history, "stop_reason_history": res.stop_reason_history,
    "partial": timed_out,
    "config": {**CONFIG, "wall_clock_min": mins, "timed_out": timed_out,
               "s_final": float(kernel.s)},
}, ckpt)
print(f"\n{'TIMED OUT' if timed_out else 'completed'} in {mins:.1f} min ({mins / 60:.2f} h)   "
      f"levels={len(levels)}")
print(f"saved -> {os.path.basename(ckpt)}  ({os.path.getsize(ckpt) / 2**20:.1f} MB)")
print(f"actual vs projected sweep cost: "
      f"{mins * 60 / max(1, sum(res.n_mcmc_history)):.1f} s vs {t_sweep:.1f} s")
print(f"beta reached: {levels[-1].beta:.4f} of 1.0   "
      f"m_eff {levels[-1].lam_eff / lam_s:.2f} of {M_TILT:.0f}   s_final={float(kernel.s):.4f}")


SMC (from scratch): N=8 R=64 N_gamma=30 gamma=[0.0009766, 1024] m=3 lam=236.0 ess_target=0.8 n_mcmc=None (adaptive, cap 40)
pCN: s0=0.400 target_acc=0.23 adapt_rate=0.1
projected 3.4 min/sweep; deadline 04:56:57
checkpointing every sweep -> partial_N8_m3_seed101_fg1.pt

*** STOPPED ON DEADLINE: deadline after 106 sweeps at beta=0.6657; 7 levels on disk
*** Rendering the levels that completed.

TIMED OUT in 367.5 min (6.13 h)   levels=7
saved -> levels_N8_m3_seed101_fg1_TIMEOUT.pt  (14.0 MB)
actual vs projected sweep cost: 208.0 s vs 205.8 s
beta reached: 0.6657 of 1.0   m_eff 2.00 of 3   s_final=0.7139


In [7]:
# --- results table --------------------------------------------------------------------------------
# uniq/N SATURATES and must never be read alone (established finding, a3_1/a3_2): systematic resampling
# makes it monotone non-increasing and it moves in steps of 1/N = 0.125 here, so it cannot distinguish
# "eight clones" from "eight nearly-identical particles". `pair/iid` below is the median pairwise
# latent distance of the cloud as a fraction of the median pairwise distance among the REFERENCES --
# which are iid draws from p, so 1.00 means "as spread out as independent samples" and a falling column
# is collapse even while uniq/N reads 1.00.
lam_eff = [l.lam_eff for l in levels]
Eqf = [float(l.fX.mean()) for l in levels]
f_sd = [float(l.fX.std()) for l in levels]
uniq = [int(l.X.unique(dim=0).shape[0]) / N for l in levels]
_ref_pair = float(torch.pdist(latent_refs.cpu().double()).median())
pair = [float(torch.pdist(l.X.cpu().double()).median()) / _ref_pair for l in levels]
# ess_history holds RAW ESS counts in [1, N]; normalize to compare against the 0.5 floor. Unlike the
# run-2 notebook these are real numbers on a truncated run too -- the kernel records state.ess.
ess = [float("nan")] + [float(e) / N for e in res.ess_history]
acc = [float("nan")] + [float(a) for a in res.acc_history]
sweeps = [0] + list(res.n_mcmc_history)
stops = ["-"] + list(res.stop_reason_history)
base_f = Eqf[0]

print(f"gamma window: [{GAMMA_LO:.4g}, {GAMMA_HI:.4g}]  (run 2 cut at {GAMMA_LO_RUN2:.4g}; the "
      f"readmitted band carries {frac_readmit:.1%} of D^2_IEM)")
print(f"lambda_s = {lam_s:.2f}  (run 2, same 32 draws, cut window: {LAM_S_RUN2:.2f})")
print(f"betas               = {[round(b, 4) for b in res.betas]}")
print(f"n_mcmc_history      = {res.n_mcmc_history}   (total {sum(res.n_mcmc_history)} sweeps)")
print(f"stop_reason_history = {res.stop_reason_history}")
print(f"\n{'k':>2} {'beta':>7} {'lam_eff':>9} {'m':>6} {'E_q[f]':>8} {'d f':>8} {'d f/sd':>7} "
      f"{'ESS/N':>6} {'acc':>6} {'uniq/N':>7} {'pair/iid':>8} {'f-sd':>8} {'swp':>4} {'stop':>8}")
for k in range(len(levels)):
    print(f"{k:2d} {levels[k].beta:7.4f} {lam_eff[k]:9.1f} {lam_eff[k] / lam_s:6.2f} "
          f"{Eqf[k]:8.4f} {Eqf[k] - base_f:+8.4f} {(Eqf[k] - base_f) / f_sd[0]:7.1f} "
          f"{ess[k]:6.2f} {acc[k]:6.2f} {uniq[k]:7.2f} {pair[k]:8.2f} {f_sd[k]:8.4f} "
          f"{sweeps[k]:4d} {stops[k]:>8}")

print(f"\nnovelty gain: E_q[f] {base_f:.4f} -> {Eqf[-1]:.4f} "
      f"({100 * (Eqf[-1] / base_f - 1):+.1f}%, {(Eqf[-1] - base_f) / f_sd[0]:.1f} sigma_p(f))")
print(f"reached beta {levels[-1].beta:.4f} of 1.0, m_eff {lam_eff[-1] / lam_s:.2f} of {M_TILT:.0f}")
print(f"diversity at the top level: uniq/N {uniq[-1]:.2f}, pair/iid {pair[-1]:.2f}, "
      f"ESS/N {ess[-1]:.2f}")
# The selection rule (CLAUDE.md): take the LARGEST lambda whose diversity stays above ~0.5. E_q[f]
# rises forever on the pCN line and is NOT the stopping signal -- degeneracy is.
_ok = [k for k in range(1, len(levels)) if uniq[k] >= 0.5 and ess[k] >= 0.5]
if _ok:
    k = _ok[-1]
    print(f"SELECTED level {k}: lam_eff={lam_eff[k]:.1f} (m={lam_eff[k] / lam_s:.2f}), "
          f"E_q[f]={Eqf[k]:.4f}, uniq/N={uniq[k]:.2f}, ESS/N={ess[k]:.2f}")
else:
    print("NO level satisfies the uniq/N >= 0.5 and ESS/N >= 0.5 selection rule -- the cloud "
          "degenerated at the first tilted level. Report that, not the last row.")
n_cap = sum(1 for r in res.stop_reason_history if r == "cap")
print(f"levels hitting the sweep cap ({MAX_SWEEPS}): {n_cap}/{len(res.stop_reason_history)}")

# --- diagnostics ----------------------------------------------------------------------------------
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
ax[0].plot(lam_eff, Eqf, "o-"); ax[0].set_ylabel(r"$E_q[f]$")
ax[0].axhline(float(f_probe.mean()), color="grey", ls=":", label=r"$E_p[f]$ (probe)")
ax[0].legend(fontsize=8); ax[0].set_title("novelty vs tilt")
ax[1].plot(lam_eff, uniq, "o-", label="uniq/N")
ax[1].plot(lam_eff, ess, "s--", label="ESS/N")
ax[1].plot(lam_eff, pair, "^-", label="pair/iid")
ax[1].axhline(0.5, color="crimson", ls=":", label="0.5 floor")
ax[1].set_ylim(0, 1.15); ax[1].legend(fontsize=8); ax[1].set_title("diversity")
ax[2].plot(lam_eff, acc, "o-", color="darkorange")
ax[2].axhline(0.23, color="grey", ls=":", label="pCN target")
ax[2].set_ylabel("acceptance"); ax[2].legend(fontsize=8); ax[2].set_title("rejuvenation health")
ax[3].plot(lam_eff, f_sd, "o-", color="darkgreen")
ax[3].set_ylabel(r"std$_q(f)$"); ax[3].set_title("f-spread (collapse uniq/N misses)")
for a in ax:
    a.set_xlabel(r"$\lambda_{eff}$"); a.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RUN_DIR, f"diagnostics_{RUN_TAG}.png"), dpi=110, bbox_inches="tight")
plt.show()

fig, a = plt.subplots(figsize=(7.5, 3.4))
cols = ["tab:red" if r == "cap" else "tab:grey" if r == "timeout" else "tab:blue"
        for r in res.stop_reason_history]
a.bar(range(len(res.betas)), res.n_mcmc_history, color=cols, alpha=0.85)
a.axhline(MAX_SWEEPS, ls="--", color="tab:red", lw=1, label=f"cap = {MAX_SWEEPS}")
a.axhline(4, ls=":", color="grey", lw=1, label="min = 4")
a.set_xticks(range(len(res.betas)))
a.set_xticklabels([f"{b:.3f}" for b in res.betas], rotation=45, fontsize=8)
a.set_xlabel(r"$\beta$"); a.set_ylabel("sweeps"); a.legend(fontsize=8); a.grid(alpha=0.3, axis="y")
a.set_title("rejuvenation effort  (red = hit cap / never decorrelated, grey = timeout)")
plt.tight_layout()
fig.savefig(os.path.join(RUN_DIR, f"effort_{RUN_TAG}.png"), dpi=110, bbox_inches="tight")
plt.show()


gamma window: [0.0009766, 1024]  (run 2 cut at 0.9713; the readmitted band carries 9.2% of D^2_IEM)
lambda_s = 78.68  (run 2, same 32 draws, cut window: 92.19)
betas               = [0.1322, 0.2342, 0.3438, 0.4463, 0.5677, 0.6657]
n_mcmc_history      = [12, 5, 7, 36, 30, 16]   (total 106 sweeps)
stop_reason_history = ['decorr', 'decorr', 'decorr', 'decorr', 'decorr', 'timeout']

 k    beta   lam_eff      m   E_q[f]      d f  d f/sd  ESS/N    acc  uniq/N pair/iid     f-sd  swp     stop
 0  0.0000       0.0   0.00   1.0100  +0.0000     0.0    nan    nan    1.00     0.97   0.0145    0        -
 1  0.1322      31.2   0.40   1.0083  -0.0017    -0.1   0.80   0.75    1.00     0.98   0.0208   12   decorr
 2  0.2342      55.3   0.70   1.0106  +0.0006     0.0   0.80   0.50    1.00     0.92   0.0168    5   decorr
 3  0.3438      81.1   1.03   1.0287  +0.0187     1.3   0.80   0.50    1.00     0.94   0.0234    7   decorr
 4  0.4463     105.3   1.34   1.0322  +0.0222     1.5   0.80   0.38    1.00   

In [8]:
# --- images ---------------------------------------------------------------------------------------
# Columns ordered by LINEAGE: walk `ancestors` backwards from the final level, so a column tracks one
# particle as the tilt strengthens. Repeated images across a row are resampler coalescence, which is
# information, not a defect. (On a timed-out run ancestors are arange, so this reduces to raw order.)
#
# THIS IS THE RUN'S ACTUAL TEST. Conclusion 1a says a readmitted Euclidean floor makes the tilt push
# particles apart in latent space rather than towards structurally novel regions, so the failure mode
# to look for is images DEGRADING (noise, washed-out colour, loss of the prompt) as lambda_eff rises,
# while E_q[f] climbs happily. Run 2's narrow window produced on-prompt dogs at every level; if this
# one does not, the window is the cause and Conclusion 1d's lower cut is vindicated empirically rather
# than only by the GMM derivation.
col_of = [None] * len(levels)
cur = torch.arange(N)
col_of[-1] = cur
for k in range(len(levels) - 1, 0, -1):
    cur = levels[k].ancestors[cur]
    col_of[k - 1] = cur

decoded = [decode(l.X[col_of[k]]) for k, l in enumerate(levels)]
print(f"decoded {len(levels)} levels x {N} particles")

nrow = len(levels)
fig, axes = plt.subplots(nrow, N, figsize=(N * 1.9, nrow * 2.05), squeeze=False)
for k in range(nrow):
    for j in range(N):
        a = axes[k][j]
        a.axis("off")
        a.imshow(decoded[k][j].permute(1, 2, 0).clamp(0, 1).numpy())
        if j == 0:
            a.text(-0.14, 0.5, f"$\\lambda_{{eff}}$={lam_eff[k]:.0f}\n$\\beta$={levels[k].beta:.3f}\n"
                               f"$E_q[f]$={Eqf[k]:.3f}",
                   transform=a.transAxes, ha="right", va="center", fontsize=7)
        if k == 0:
            a.set_title(f"col {j}", fontsize=7)
fig.suptitle(f"RUN fg1 -- FULL $\\gamma$ window $[2^{{-10}}, 2^{{10}}]$ "
             f"(R={R}, $N_\\gamma$={N_GAMMA}): m={M_TILT:.0f}, "
             f"$\\lambda_{{max}}$={LAM_MAX:.0f}   prompt {PROMPT!r}, N={N}\n"
             f"rows: $\\lambda_{{eff}}$ ascending (top = untilted $p$)   columns: lineage", fontsize=12)
plt.tight_layout(rect=(0.05, 0, 1, 0.965))
fig.savefig(GRID_PNG, dpi=110, bbox_inches="tight")
plt.show()

# Untilted anchor vs most tilted, enlarged, for a direct visual read on what the tilt did.
for k, label in ((0, "UNTILTED $\\lambda_{eff}=0$ (samples from $p$)"),
                 (len(levels) - 1, f"MOST TILTED $\\lambda_{{eff}}$={lam_eff[-1]:.0f} "
                                   f"(m={lam_eff[-1] / lam_s:.1f})")):
    fig, axes = plt.subplots(1, N, figsize=(N * 1.9, 2.6), squeeze=False)
    for j in range(N):
        a = axes[0][j]
        a.axis("off")
        a.imshow(decoded[k][j].permute(1, 2, 0).clamp(0, 1).numpy())
        a.set_title(f"f={float(levels[k].fX[col_of[k][j]]):.4f}", fontsize=7)
    fig.suptitle(label, fontsize=12)
    plt.tight_layout(rect=(0, 0, 1, 0.90))
    fig.savefig(os.path.join(RUN_DIR, f"level{k:02d}_{RUN_TAG}.png"), dpi=110, bbox_inches="tight")
    plt.show()

# Per-image PNGs for full-resolution inspection. ~0.44 MB each; N=8 keeps a full ladder well under the
# ~50 MB/leg the sweep directory's .gitignore budgets for.
from PIL import Image

os.makedirs(DECODED_DIR, exist_ok=True)
for k in range(len(levels)):
    for j in range(N):
        arr = (decoded[k][j].permute(1, 2, 0).clamp(0, 1) * 255).round().to(torch.uint8).numpy()
        Image.fromarray(arr).save(
            os.path.join(DECODED_DIR, f"lvl{k:02d}_lam{lam_eff[k]:.0f}_col{j}.png"))
print(f"wrote {len(levels) * N} PNGs to {os.path.basename(DECODED_DIR)}/ and "
      f"{os.path.basename(GRID_PNG)}")

# Latent-space health check. A Euclidean push-away tilt inflates the latent norm; the VAE only decodes
# sensibly near the scale it was trained at, so a rising std here is the quantitative version of
# "the images went bad" and is worth reading even if the grid looks fine.
print(f"\nlatent std by level (references: {S:.4f}):")
for k in range(len(levels)):
    print(f"  lvl {k:2d}  lam_eff={lam_eff[k]:8.1f}  std={float(levels[k].X.std()):.4f}  "
          f"mean|x|={float(levels[k].X.abs().mean()):.4f}")


decoded 7 levels x 8 particles
wrote 56 PNGs to decoded_fg1/ and fullgamma_grid_fg1.png

latent std by level (references: 1.0147):
  lvl  0  lam_eff=     0.0  std=0.9793  mean|x|=0.7699
  lvl  1  lam_eff=    31.2  std=0.9887  mean|x|=0.7829
  lvl  2  lam_eff=    55.3  std=0.9217  mean|x|=0.7304
  lvl  3  lam_eff=    81.1  std=0.9709  mean|x|=0.7670
  lvl  4  lam_eff=   105.3  std=0.9626  mean|x|=0.7589
  lvl  5  lam_eff=   134.0  std=0.9636  mean|x|=0.7624
  lvl  6  lam_eff=   157.1  std=0.9370  mean|x|=0.7441


## Takeaways

*(To be written after the run. Per the repo convention this cell is the authoritative statement of what
was learned; the tables above are evidence, not conclusions.)*

Questions this run must answer, in the order the evidence arrives:

1. **How much does the readmitted band carry?** — cell 5, `frac_readmitted`. Available ~50 min in,
   before any sweeping, so it stands even if the SMC is truncated.
2. **Does Conclusion 1a transfer to FLUX?** — cell 5's measured/predicted ratio against
   $\lVert \Delta x \rVert^{2} / (\gamma S^{2} + 1)^{2}$, no fitted constant.
3. **Is $N_{\gamma} = 30$ still enough over 20 octaves?** — cell 5's peak-interval share. This is the
   one knob the run knowingly leaves under-justified (Concl. 2's $N_{\gamma} \ge 25$ was calibrated on
   the *cut* window); a follow-up at $N_{\gamma} = 60$ is the clean bound.
4. **Did $\lambda_{s}$ move, and by how much?** — cell 4, against run 2's 92.195 on the identical 32
   held-out draws. Concl. 1b predicts the window rescales the whole distance.
5. **What did the tilt actually produce?** — cell 8. The falsifiable prediction is that a dominant
   Euclidean floor pushes particles off-manifold: decoded images degrade while $E_{q}[f]$ rises, and
   the latent std climbs above the reference $S \approx 1.015$. Run 2's narrow window kept clean
   on-prompt dogs at every level, so a divergence here is attributable to the window alone — the
   reference latents, seeds, generator and all other knobs are bit-identical.

Record the selected level by the standing rule (largest $\lambda$ with `uniq/N` and ESS/N above ~0.5),
not the last row, and note `stop_reason_history` before trusting any level that stopped on `timeout`.
